In [ ]:
# Extra libraries installed in /scratch
import sys
sys.path.insert(0, '/scratch/network/ih2422/lib')

# Importing functions from other files
from narratives_prep import *

In [ ]:
# Fetch data for 21st year story (hence abbreviated as "twenty")
twenty_segments   = fetch_segments("21styear")
twenty_subjects = fetch_subject_list("21styear")
print(f"{len(twenty_subjects)} twenty subjects: {twenty_subjects}")

# Fetch data for tunnel story
tunnel_segments   = fetch_segments("tunnel")
tunnel_subjects = fetch_subject_list("tunnel", exclude_subjects=["sub-004", "sub-013"])
print(f"{len(tunnel_subjects)} tunnel subjects: {tunnel_subjects}")

### 1. Place Annotation (with Claude)

In [ ]:
# Tunnel story places
tunnel_places = annotate_places(
    "Tunnel Under the World", fetch_whisperx_transcript("tunnel"),
    ["guy_home", "bus", "office"],
    out_dir / "tunnel_places.tsv"
)

# Twenty story places
twenty_places = annotate_places(
    "21st Year", fetch_whisperx_transcript("21styear"),
    ["clara_home", "steven_home", "cab"],
    out_dir / "twenty_places.tsv"
)

In [ ]:
tunnel_colors = {"guy_home": "#4C72B0", "bus": "#DD8452", "office": "#55A868", "cellar": "#8C564B", "other": "#bbbbbb"}
twenty_colors  = {"clara_home": "#C44E52", "steven_home": "#8172B2", "cab": "#64B5CD", "other": "#bbbbbb"}

tunnel_aligned = align_tsv_to_whisperx(Path("tunnel_places.tsv"), tunnel_segments, fetch_whisperx_transcript("tunnel"))
twenty_aligned = align_tsv_to_whisperx(Path("twenty_places.tsv"), twenty_segments, fetch_whisperx_transcript("21styear"))

plot_place_line(tunnel_aligned, "Tunnel Under the World — place timeline",
                ["guy_home", "bus", "office", "cellar"], tunnel_colors)
plot_place_line(twenty_aligned, "21st Year — place timeline",
                ["clara_home", "steven_home", "cab"], twenty_colors)

In [ ]:
tunnel_aligned

In [ ]:
fetch_whisperx_transcript("21styear")

In [ ]:
twenty_aligned

### Single-Subject Analysis: Tunnel Story

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ISC-based attention filter (Hasson et al., 2004)
from nilearn import datasets

_mni = datasets.load_mni152_brain_mask(resolution=4)
isc_scores, tunnel_subjects_filtered = isc_filter(
    tunnel_subjects, "tunnel", _mni,
    title="Tunnel Story — Inter-Subject Correlation"
)


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import LeaveOneOut
from nilearn.maskers import NiftiMasker
from nilearn import datasets
from nilearn.image import resample_to_img
import numpy as np

TR  = 1.5
HRF_LAG_TRS = 3

mni_mask = datasets.load_mni152_brain_mask(resolution=4)
schaefer  = datasets.fetch_atlas_schaefer_2018(n_rois=400, yeo_networks=17, resolution_mm=2)
labels    = [l.decode() if isinstance(l, bytes) else l for l in schaefer.labels]

labeled_places = ["guy_home", "bus", "office", "cellar"]
events = [r for r in tunnel_aligned if r["place"] in labeled_places]

def event_pattern(event, bold, TR, lag_trs):
    start_tr = int(event["start"] / TR) + lag_trs
    end_tr   = int(event["end"]   / TR) + lag_trs
    start_tr = max(0, min(start_tr, len(bold) - 1))
    end_tr   = max(start_tr + 1, min(end_tr, len(bold)))
    return bold[start_tr:end_tr].mean(axis=0)

# Encode labels once — same for every subject
le = LabelEncoder()
y_encoded = le.fit_transform([e["place"] for e in events])

# Build atlas and ROI masks once — atlas is subject-independent
ref_masker = NiftiMasker(mask_img=mni_mask, standardize=True)
ref_masker.fit()
schaefer_img  = resample_to_img(schaefer.maps, ref_masker.mask_img_, interpolation='nearest')
schaefer_data = schaefer_img.get_fdata().astype(int)
mask_data     = ref_masker.mask_img_.get_fdata().astype(bool)

parcel_label = {i + 1: lbl for i, lbl in enumerate(labels)}

roi_masks = {}
for roi, searches in [("PHC", ["PHC"]), ("Rsp", ["Rsp"]), ("OPA", ["DorsAttnA_TempOcc", "DorsAttnB_TempOcc"]),
                       ("mPFC", ["DefaultA_PFCm"]), ("AG", ["DefaultB_IPL", "DefaultC_IPL"]),
                       ("PMC", ["_pCun"])]:
    vol = np.zeros(schaefer_data.shape, dtype=bool)
    for idx, lbl in parcel_label.items():
        if any(s in lbl for s in searches):
            vol |= (schaefer_data == idx)
    roi_masks[roi] = vol & mask_data

subject_acc = {}

for sub_i in tunnel_subjects_filtered:
    data_i = load_bold_masked(sub_i, "tunnel", mni_mask, standardize=True)
    if data_i is None:
        print(f"{sub_i}  — file missing, skipping")
        continue

    stimulus_duration = data_i.shape[0] * TR

    X_i = np.vstack([event_pattern(e, data_i, TR, HRF_LAG_TRS) for e in events])
    y_i = y_encoded

    pipe_wb = Pipeline([
        ("scaler", StandardScaler()),
        ("svm",    LinearSVC(max_iter=10000))
    ])

    loo = LeaveOneOut()
    preds_wb = np.zeros(len(X_i), dtype=int)
    for train_idx, test_idx in loo.split(X_i):
        pipe_wb.fit(X_i[train_idx], y_i[train_idx])
        preds_wb[test_idx[0]] = pipe_wb.predict(X_i[test_idx])[0]
    acc_wb = np.mean(preds_wb == y_i)

    # ROI accuracies (no feature selection — ROIs already small)
    roi_accs = {}
    for roi, roi_mask in roi_masks.items():
        voxels   = data_i[:, roi_mask[mask_data]]
        X_roi    = np.vstack([event_pattern(e, voxels, TR, HRF_LAG_TRS) for e in events])
        pipe_roi = Pipeline([
            ("scaler", StandardScaler()),
            ("svm", LinearSVC(max_iter=10000))
        ])
        preds_roi = np.zeros(len(X_roi), dtype=int)
        for train_idx, test_idx in loo.split(X_roi):
            pipe_roi.fit(X_roi[train_idx], y_i[train_idx])
            preds_roi[test_idx[0]] = pipe_roi.predict(X_roi[test_idx])[0]
        roi_accs[roi] = np.mean(preds_roi == y_i)

    subject_acc[sub_i] = {"wb": acc_wb, "PHC": roi_accs["PHC"], "Rsp": roi_accs["Rsp"], "OPA": roi_accs["OPA"], "mPFC": roi_accs["mPFC"], "AG": roi_accs["AG"], "PMC": roi_accs["PMC"]}
    print(f"{sub_i}  dur={stimulus_duration:.1f}s  whole-brain={acc_wb:.2f}  PHC={roi_accs['PHC']:.2f}  Rsp={roi_accs['Rsp']:.2f}  OPA={roi_accs['OPA']:.2f}  mPFC={roi_accs['mPFC']:.2f}  AG={roi_accs['AG']:.2f}  PMC={roi_accs['PMC']:.2f}")

chance = 1 / len(le.classes_)
mean_wb   = np.mean([v["wb"]   for v in subject_acc.values()])
mean_phc  = np.mean([v["PHC"]  for v in subject_acc.values()])
mean_rsp  = np.mean([v["Rsp"]  for v in subject_acc.values()])
mean_opa  = np.mean([v["OPA"]  for v in subject_acc.values()])
mean_mpfc = np.mean([v["mPFC"] for v in subject_acc.values()])
mean_ag   = np.mean([v["AG"]   for v in subject_acc.values()])
mean_pmc  = np.mean([v["PMC"]  for v in subject_acc.values()])
print(f"\nMean  whole-brain={mean_wb:.2f}  PHC={mean_phc:.2f}  Rsp={mean_rsp:.2f}  OPA={mean_opa:.2f}  mPFC={mean_mpfc:.2f}  AG={mean_ag:.2f}  PMC={mean_pmc:.2f}  (chance={chance:.2f})")
print(f"Above chance (wb): {sum(v['wb'] > chance for v in subject_acc.values())}/{len(subject_acc)} subjects")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

wb_accs  = [v["wb"]  for v in subject_acc.values()]
phc_accs = [v["PHC"] for v in subject_acc.values()]
opa_accs = [v["OPA"] for v in subject_acc.values()]
chance   = 1 / 4

means = [np.mean(wb_accs), np.mean(phc_accs), np.mean(opa_accs)]
sems  = [stats.sem(wb_accs), stats.sem(phc_accs), stats.sem(opa_accs)]

fig, ax = plt.subplots(figsize=(5, 4))
x = np.arange(3)
ax.bar(x, means, yerr=sems, capsize=5,
       color=["#55A868", "#4C72B0", "#DD8452"], width=0.5, error_kw=dict(elinewidth=1.5))
ax.axhline(chance, color="black", linestyle="--", linewidth=1, label=f"Chance ({chance:.2f})")
ax.set_xticks(x)
ax.set_yticks(np.arange(0, 1.1, 0.1))
ax.set_xticklabels(["Whole Brain", "PPA", "OPA"])
ax.set_ylabel("Accuracy (LOO-CV)")
ax.set_title("Tunnel Story — Decoding (n={})".format(len(subject_acc)))
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.model_selection import permutation_test_score
import numpy as np
import matplotlib.pyplot as plt

n_permutations = 10

_pipe_perm = Pipeline([
    ("svm", LinearSVC(max_iter=10000))
])

obs_scores      = []
perm_scores_all = []

for sub_i in tunnel_subjects_filtered:
    data_i = load_bold_masked(sub_i, "tunnel", mni_mask, standardize=True)
    if data_i is None:
        continue
    X_i = np.vstack([event_pattern(e, data_i, TR, HRF_LAG_TRS) for e in events])
    score, perm_scores, p_value = permutation_test_score(
        _pipe_perm, X_i, y_encoded,
        cv=LeaveOneOut(), n_permutations=n_permutations,
        scoring='accuracy', n_jobs=-1, random_state=42
    )
    obs_scores.append(score)
    perm_scores_all.append(perm_scores)
    print(f"{sub_i}  obs={score:.3f}  p={p_value:.4f}")

null_dist = np.stack(perm_scores_all).mean(axis=0)
mean_obs  = np.mean(obs_scores)
p_pooled  = (null_dist >= mean_obs).mean()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(null_dist, bins=30, color='#aaaaaa', edgecolor='white', label='Null (permuted labels)')
ax.axvline(mean_obs, color='#d62728', linewidth=2, label=f'Observed mean = {mean_obs:.3f}')
ax.axvline(1/3, color='k', linestyle='--', linewidth=1, label='Chance (0.33)')
ax.set_xlabel('Accuracy')
ax.set_ylabel('Count')
ax.set_title(f'Permutation test — Tunnel story, whole-brain (n={len(obs_scores)} subjects)\n'
             f'p = {p_pooled:.4f} ({n_permutations} permutations)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import permutation_test_score
import numpy as np
import matplotlib.pyplot as plt

n_permutations = 50

_pipe_perm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(max_iter=10000))
])

obs_scores      = []
perm_scores_all = []

for sub_i in tunnel_subjects_filtered:
    data_i = load_bold_masked(sub_i, "tunnel", mni_mask, standardize=True)
    if data_i is None:
        continue
    X_i = np.vstack([event_pattern(e, data_i, TR, HRF_LAG_TRS) for e in events])
    score, perm_scores, p_value = permutation_test_score(
        _pipe_perm, X_i, y_encoded,
        cv=LeaveOneOut(), n_permutations=n_permutations,
        scoring='accuracy', n_jobs=-1, random_state=42
    )
    obs_scores.append(score)
    perm_scores_all.append(perm_scores)
    print(f"{sub_i}  obs={score:.3f}  p={p_value:.4f}")

null_dist = np.stack(perm_scores_all).mean(axis=0)
mean_obs  = np.mean(obs_scores)
p_pooled  = (null_dist >= mean_obs).mean()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(null_dist, bins=30, color='#aaaaaa', edgecolor='white', label='Null (permuted labels)')
ax.axvline(mean_obs, color='#d62728', linewidth=2, label=f'Observed mean = {mean_obs:.3f}')
ax.axvline(1/3, color='k', linestyle='--', linewidth=1, label='Chance (0.33)')
ax.set_xlabel('Accuracy')
ax.set_ylabel('Count')
ax.set_title(f'Permutation test — Tunnel story, whole-brain (n={len(obs_scores)} subjects)\n'
             f'p = {p_pooled:.4f} ({n_permutations} permutations)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
events

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Collect per-subject RSA matrices
rsa_list = []
for sub_i in tunnel_subjects_filtered:
    data_i = load_bold_masked(sub_i, 'tunnel', mni_mask, standardize=True)
    if data_i is None:
        continue
    X_i = np.vstack([event_pattern(e, data_i, TR, HRF_LAG_TRS) for e in events])
    rsa_list.append(np.corrcoef(X_i))

mean_rsa = np.mean(rsa_list, axis=0)

# Sort by place type; duplicates share the same label
place_labels  = [e['place'] for e in events]
sort_order    = sorted(range(len(events)), key=lambda i: place_labels[i])
sorted_labels = [place_labels[i] for i in sort_order]
rsa_sorted    = mean_rsa[np.ix_(sort_order, sort_order)]

# Group boundary positions
counts, prev = [], None
for lbl in sorted_labels:
    if lbl != prev:
        counts.append(1); prev = lbl
    else:
        counts[-1] += 1

fig, ax = plt.subplots(figsize=(7, 6))
img = ax.imshow(rsa_sorted, cmap='RdBu_r', vmin=-1, vmax=1)

cum = 0
for c in counts[:-1]:
    cum += c
    ax.axhline(cum - 0.5, color='k', linewidth=1.5)
    ax.axvline(cum - 0.5, color='k', linewidth=1.5)

ax.set_xticks(range(len(sorted_labels)))
ax.set_yticks(range(len(sorted_labels)))
ax.set_xticklabels(sorted_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(sorted_labels, fontsize=9)
fig.colorbar(img, label='Pearson r')
ax.set_title('RSA — Tunnel story events (mean across subjects, correct timing)')
plt.tight_layout()
plt.show()

In [ ]:
raise NotImplementedError("Stop here for now")

### Single-Subject Analysis: Twenty Story

In [ ]:
# ISC-based attention filter (Hasson et al., 2004)
from nilearn import datasets

_mni = datasets.load_mni152_brain_mask(resolution=4)
isc_scores, twenty_subjects_filtered = isc_filter(
    twenty_subjects, "21styear", _mni,
    title="Twenty Story — Inter-Subject Correlation"
)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import LeaveOneOut
from nilearn.maskers import NiftiMasker
from nilearn import datasets
from nilearn.image import resample_to_img
import numpy as np

TR_20 = 1.5
HRF_LAG_TRS_20 = 3

mni_mask_20  = datasets.load_mni152_brain_mask(resolution=4)
schaefer_20  = datasets.fetch_atlas_schaefer_2018(n_rois=400, yeo_networks=17, resolution_mm=2)
labels_20    = [l.decode() if isinstance(l, bytes) else l for l in schaefer_20.labels]

labeled_places_20 = ["clara_home", "steven_home"]
events_20 = [r for r in twenty_aligned if r["place"] in labeled_places_20]

def event_pattern_20(event, bold, TR, lag_trs):
    start_tr = int(event["start"] / TR) + lag_trs
    end_tr   = int(event["end"]   / TR) + lag_trs
    start_tr = max(0, min(start_tr, len(bold) - 1))
    end_tr   = max(start_tr + 1, min(end_tr, len(bold)))
    return bold[start_tr:end_tr].mean(axis=0)

le_20 = LabelEncoder()
y_20  = le_20.fit_transform([e["place"] for e in events_20])

ref_masker_20 = NiftiMasker(mask_img=mni_mask_20, standardize=True)
ref_masker_20.fit()
schaefer_img_20  = resample_to_img(schaefer_20.maps, ref_masker_20.mask_img_, interpolation='nearest')
schaefer_data_20 = schaefer_img_20.get_fdata().astype(int)
mask_data_20     = ref_masker_20.mask_img_.get_fdata().astype(bool)
parcel_label_20  = {i + 1: lbl for i, lbl in enumerate(labels_20)}

roi_masks_20 = {}
for roi, searches in [("PHC", ["PHC"]), ("Rsp", ["Rsp"]), ("OPA", ["DorsAttnA_TempOcc", "DorsAttnB_TempOcc"]),
                       ("mPFC", ["DefaultA_PFCm"]), ("AG", ["DefaultB_IPL", "DefaultC_IPL"]),
                       ("PMC", ["_pCun"])]:
    vol = np.zeros(schaefer_data_20.shape, dtype=bool)
    for idx, lbl in parcel_label_20.items():
        if any(s in lbl for s in searches):
            vol |= (schaefer_data_20 == idx)
    roi_masks_20[roi] = vol & mask_data_20

subject_acc_20 = {}

for sub_i in twenty_subjects_filtered:
    data_i = load_bold_masked(sub_i, "21styear", mni_mask_20, standardize=True)
    if data_i is None:
        print(f"{sub_i}  — file missing, skipping")
        continue

    X_i = np.vstack([event_pattern_20(e, data_i, TR_20, HRF_LAG_TRS_20) for e in events_20])

    pipe_wb = Pipeline([
        ("scaler", StandardScaler()),
        ("svm",    LinearSVC(max_iter=10000))
    ])
    loo = LeaveOneOut()
    preds_wb = np.zeros(len(X_i), dtype=int)
    for train_idx, test_idx in loo.split(X_i):
        pipe_wb.fit(X_i[train_idx], y_20[train_idx])
        preds_wb[test_idx[0]] = pipe_wb.predict(X_i[test_idx])[0]
    acc_wb = np.mean(preds_wb == y_20)

    roi_accs = {}
    for roi, roi_mask in roi_masks_20.items():
        voxels    = data_i[:, roi_mask[mask_data_20]]
        X_roi     = np.vstack([event_pattern_20(e, voxels, TR_20, HRF_LAG_TRS_20) for e in events_20])
        pipe_roi  = Pipeline([("scaler", StandardScaler()), ("svm", LinearSVC(max_iter=10000))])
        preds_roi = np.zeros(len(X_roi), dtype=int)
        for train_idx, test_idx in loo.split(X_roi):
            pipe_roi.fit(X_roi[train_idx], y_20[train_idx])
            preds_roi[test_idx[0]] = pipe_roi.predict(X_roi[test_idx])[0]
        roi_accs[roi] = np.mean(preds_roi == y_20)

    subject_acc_20[sub_i] = {"wb": acc_wb, "PHC": roi_accs["PHC"], "Rsp": roi_accs["Rsp"],
                              "OPA": roi_accs["OPA"], "mPFC": roi_accs["mPFC"],
                              "AG": roi_accs["AG"], "PMC": roi_accs["PMC"]}
    print(f"{sub_i}  whole-brain={acc_wb:.2f}  PHC={roi_accs['PHC']:.2f}  Rsp={roi_accs['Rsp']:.2f}  OPA={roi_accs['OPA']:.2f}  mPFC={roi_accs['mPFC']:.2f}  AG={roi_accs['AG']:.2f}  PMC={roi_accs['PMC']:.2f}")

chance_20 = 1 / len(le_20.classes_)
mean_wb_20   = np.mean([v["wb"]   for v in subject_acc_20.values()])
mean_phc_20  = np.mean([v["PHC"]  for v in subject_acc_20.values()])
mean_rsp_20  = np.mean([v["Rsp"]  for v in subject_acc_20.values()])
mean_opa_20  = np.mean([v["OPA"]  for v in subject_acc_20.values()])
mean_mpfc_20 = np.mean([v["mPFC"] for v in subject_acc_20.values()])
mean_ag_20   = np.mean([v["AG"]   for v in subject_acc_20.values()])
mean_pmc_20  = np.mean([v["PMC"]  for v in subject_acc_20.values()])
print(f"\nMean  whole-brain={mean_wb_20:.2f}  PHC={mean_phc_20:.2f}  Rsp={mean_rsp_20:.2f}  OPA={mean_opa_20:.2f}  mPFC={mean_mpfc_20:.2f}  AG={mean_ag_20:.2f}  PMC={mean_pmc_20:.2f}  (chance={chance_20:.2f})")
print(f"Above chance (wb): {sum(v['wb'] > chance_20 for v in subject_acc_20.values())}/{len(subject_acc_20)} subjects")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

wb_accs  = [v["wb"]  for v in subject_acc_20.values()]
phc_accs = [v["PHC"] for v in subject_acc_20.values()]
rsp_accs = [v["Rsp"] for v in subject_acc_20.values()]
chance   = 1 / 2

means = [np.mean(wb_accs), np.mean(phc_accs), np.mean(rsp_accs)]
sems  = [stats.sem(wb_accs), stats.sem(phc_accs), stats.sem(rsp_accs)]

fig, ax = plt.subplots(figsize=(5, 4))
x = np.arange(3)
ax.bar(x, means, yerr=sems, capsize=5,
       color=["#55A868", "#4C72B0", "#DD8452"], width=0.5, error_kw=dict(elinewidth=1.5))
ax.axhline(chance, color="black", linestyle="--", linewidth=1, label=f"Chance ({chance:.2f})")
ax.set_xticks(x)
ax.set_xticklabels(["Whole Brain", "PPA (PHC)", "RSC (Rsp)"])
ax.set_ylabel("Accuracy (LOO-CV)")
ax.set_title("21st Year Story — Decoding (n={})".format(len(subject_acc_20)))
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from nilearn import datasets

TR_rsa      = 1.5
HRF_LAG_rsa = 3

def _event_pattern(event, bold, TR, lag):
    start_tr = int(event["start"] / TR) + lag
    end_tr   = int(event["end"]   / TR) + lag
    start_tr = max(0, min(start_tr, len(bold) - 1))
    end_tr   = max(start_tr + 1, min(end_tr, len(bold)))
    return bold[start_tr:end_tr].mean(axis=0)

_mni_rsa = datasets.load_mni152_brain_mask(resolution=4)

# Events grouped: all clara first, then all steven
_rsa_places = ["clara_home", "steven_home"]
_events_rsa = sorted([r for r in twenty_aligned if r["place"] in _rsa_places],
                     key=lambda r: _rsa_places.index(r["place"]))
n_ev = len(_events_rsa)
print(f"{n_ev} events ({sum(e['place']=='clara_home' for e in _events_rsa)} clara, "
      f"{sum(e['place']=='steven_home' for e in _events_rsa)} steven)")

# Load event patterns for every subject, z-scored across segments
_all_pats = {}
for sub in twenty_subjects:
    data = load_bold_masked(sub, "21styear", _mni_rsa, standardize=True)
    if data is None:
        continue
    pats = np.vstack([_event_pattern(e, data, TR_rsa, HRF_LAG_rsa) for e in _events_rsa])
    # z-score each voxel across segments (removes mean activation per voxel)
    pats = (pats - pats.mean(axis=0)) / (pats.std(axis=0) + 1e-8)
    _all_pats[sub] = pats
    print(f"  {sub} loaded")

_subs    = list(_all_pats.keys())
_dat_arr = np.stack([_all_pats[s] for s in _subs])  # (n_subs, n_events, n_voxels)

# Leave-one-out cross-subject RSA (Finn et al. 2022 approach):
# For each subject i, correlate their patterns against the LOO group mean
_rdm_z = np.zeros((n_ev, n_ev))
for i in range(len(_subs)):
    loo_mean = np.delete(_dat_arr, i, axis=0).mean(axis=0)  # (n_events, n_voxels)
    subj     = _dat_arr[i]

    sz = subj    - subj.mean(axis=1, keepdims=True)
    gz = loo_mean - loo_mean.mean(axis=1, keepdims=True)
    sz /= np.linalg.norm(sz, axis=1, keepdims=True) + 1e-8
    gz /= np.linalg.norm(gz, axis=1, keepdims=True) + 1e-8

    rdm = sz @ gz.T  # (n_events, n_events)
    _rdm_z += np.arctanh(np.clip(rdm, -0.9999, 0.9999))

mean_rdm = np.tanh(_rdm_z / len(_subs))

# Plot
_labels     = [e["place"].replace("_home", "") for e in _events_rsa]
_colors     = {"clara": "#C44E52", "steven": "#8172B2"}
_bar_colors = [_colors[l] for l in _labels]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(mean_rdm, cmap="RdBu_r", vmin=-0.5, vmax=0.5)
ax.set_xticks(range(n_ev))
ax.set_yticks(range(n_ev))
ax.set_xticklabels(_labels, rotation=90, fontsize=7)
ax.set_yticklabels(_labels, fontsize=7)
for tick, c in zip(ax.get_xticklabels(), _bar_colors):
    tick.set_color(c)
for tick, c in zip(ax.get_yticklabels(), _bar_colors):
    tick.set_color(c)
plt.colorbar(im, ax=ax, label="Pearson r (Fisher z avg)")
ax.set_title(f"RSA — twenty story, LOO cross-subject (n={len(_subs)})")
legend = [mpatches.Patch(color=v, label=k) for k, v in _colors.items()]
ax.legend(handles=legend, loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()
